<div class="uic-course-header">
<span class="uic-course-code">CS 494</span>
<span class="uic-course-title">Edge Computing Systems</span>
<span class="uic-course-meta"><span class="uic-notebook-label">Lab 10</span><span class="uic-course-term">Fall 2026</span><span class="uic-course-unit">University of Illinois Chicago · Computer Science</span></span>
</div>

# Failure Injection · Edge-Computing Notebook

Every edge node fails in the field sooner or later: the network drops, the power goes out mid-write, the disk fills, a sensor returns garbage, the clock jumps, the kernel kills a process that used too much memory. You would rather cause each of those yourself, today, while you are watching, than meet it for the first time on a device you cannot reach.

In this lab you run **one sensor feeding two loggers** that do the same job: a **naive** logger written the way code usually looks when it works on the bench, and a **hardened** logger written for the field. Then you inject six failures, one at a time, into your own containers, and record what each logger actually did.

```text
sensor --local link--> logger-naive    --MQTT uplink--> broker (stands in for the cloud)
       --local link--> logger-hardened --MQTT uplink--> broker
                       each logger writes SQLite on its own 16 MB RAM disk
```

You will:

- build a small sensor and an edge logger with a naive and a hardened mode
- run both side by side, so every drill shows both behaviors at once
- inject six faults: **NaN from the sensor**, **network outage**, **OOM kill**, **power cut mid-write**, **disk full**, and **clock jump**
- end with an **expected / observed / gap** table, which is exactly what goes in your final report and your Milestone 2 evidence

The main idea: a system is field-ready when you have watched it fail on purpose and it failed the way you designed it to.

## How this notebook works · cells and a Jupyter terminal

Steps that run and return (writing files, `podman compose up -d`, each drill) are ordinary **notebook cells** · just run them here. A drill cell runs for between 20 seconds and about 3 minutes and prints both loggers every few seconds while it runs.

Two steps belong in a terminal: a **live view** of both loggers that never returns on its own, and the **power cut**, where you pull the plug yourself. For those, this notebook writes small scripts into your lab folder.

Two labels are used throughout:

- **[Notebook cell]** · run the cell right here.
- **[Jupyter terminal]** · open a Jupyter terminal (**File ▸ New ▸ Terminal**, or the Terminal tile in the Launcher) and run the given command there. You can open several terminals at once.

### One config, everywhere

Terminal shells do not share the notebook kernel's variables. So the setup cell writes `~/faultLab/labEnv.sh`, and every helper script starts with `source ~/faultLab/labEnv.sh`, so the scripts always agree with the notebook. The whole stack talks over an internal Compose network, so this lab needs **no host ports** and cannot collide with other students.

**Time:** about 35 minutes, of which roughly 8 are drills you watch run. **Requirements:** a Python 3 kernel on the DGX Spark, Podman + Compose available to your user. This lab reuses the MQTT pattern from the *MQTT / HTTP / WebSocket* lab and the restart policy from the *Fleet* lab.

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, preflight checks, and checkpoints.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeNotebook"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
# The toolkit is republished during the term. If this kernel already imported an
# older copy, `from labHelpers import *` would silently reuse it and you would
# get a NameError for anything added since, so reload before importing.
import importlib
if "labHelpers" in sys.modules:
    importlib.reload(sys.modules["labHelpers"])
from labHelpers import *

---
## Part 0 · Before You Begin

You are working on a shared **DGX Spark** edge device · and this notebook runs there too. Every cell in this exercise runs on the DGX Spark, not on your own computer.

Breaking things on a shared machine needs rules. Every fault in this lab is injected into **your own containers only**:

- you never touch the host network: the "network outage" stops **your** broker container
- you never fill a shared disk: each logger writes to its own **16 MB RAM disk**, and that is what you fill
- you never change the system clock (you could not anyway): the "clock jump" shifts the time **inside the logger**
- the memory limit that triggers the OOM kill is on **your** logger container, 96 MB
- you cannot pull the power on the DGX Spark: the "power cut" kills **your** logger process with `SIGKILL`, and the notebook says plainly what that does and does not prove

Everything carries your account name: the Compose project is `${USER}-faults`, and the cells below already use it.

---
## Part 1 · Set Up the Project

**[Notebook cell]** The cell below creates `~/faultLab`, writes `labEnv.sh`, and exports `FAULT_PROJECT` · the `${USER}-faults` Compose project name used everywhere. (The `PORT` it derives is a spare; this lab publishes no host ports.)

In [ ]:
import os

# No host ports are needed - the stack talks over an internal Compose network.
# FAULT_PROJECT names your Compose project so it never collides with others.
labConfig = setupLab(
    "faultLab",
    ports={"PORT": 20000},   # spare; unused by this lab
    extraEnv={"FAULT_PROJECT": f"{os.environ.get('USER', 'student01')}-faults"},
)

### Preflight · check your environment

Run this once at the start and fix any failing check before continuing. The third check matters for the OOM drill: a container memory limit only works if the kernel lets your rootless Podman manage memory (cgroup v2 with the memory controller delegated to your account).

In [ ]:
import os

def memoryLimitsEnforced():
    """Probe: cgroup v2 with the memory controller delegated to this user."""
    def probe():
        out, code = runShell(["podman", "info", "--format",
                              "{{.Host.CgroupsVersion}} {{.Host.CgroupControllers}}"])
        ok = code == 0 and out.startswith("v2") and "memory" in out
        return ok, out.strip()[:80]
    return probe

preflight(
    [
        check("podman reachable", podmanReachable(),
              hint="Podman is not reachable from your account. Try "
                   "<code>podman info</code> in a Terminal; if it fails, ask your instructor.",
              link="https://docs.podman.io/en/latest/markdown/podman.1.html",
              linkText="Podman command reference"),
        check("podman compose available", composeAvailable(),
              hint="The stack is wired together with Compose. If <code>podman compose version</code> "
                   "fails in a Terminal, ask your instructor.",
              link="https://docs.docker.com/compose/", linkText="Docker Compose docs"),
        check("container memory limits are enforced", memoryLimitsEnforced(),
              hint="Without cgroup v2 memory delegation a <code>mem_limit</code> is silently "
                   "ignored and the OOM drill cannot work. Ask your instructor.",
              link="https://docs.podman.io/en/latest/markdown/podman-run.1.html#memory-m-number-unit",
              linkText="podman run --memory"),
    ],
    infoRows=[("your USER", os.environ.get("USER", "?")),
              ("your Compose project", os.environ.get("FAULT_PROJECT", "?"))],
)

**[Notebook cell]** Create the project structure:

In [ ]:
!mkdir -p ~/faultLab/app ~/faultLab/mosquitto ~/faultLab/control ~/faultLab/status ~/faultLab/results
%cd ~/faultLab

The structure will become:

```text
faultLab/
  app/
    sensor.py        the sensor: a reading 5 times a second, over a local link
    logger.py        the edge logger: MODE=naive or MODE=hardened
    Dockerfile
  mosquitto/
    mosquitto.conf   the broker that stands in for the cloud
  compose.yaml
  drills.py          lab instrumentation: watch, measure, record
  watchLoggers.sh    live view, for a terminal
  powerCut.sh        the power-cut drill, for a terminal
  control/           fault switches the drills write
  status/            what each service reports, twice a second
  results/           one file per drill, and the final table
```

---
## Part 2 · The Failures Worth Injecting

This is the table from the lecture. The middle column is what a well-built system does; the right column is what student systems, and plenty of professional ones, do the first time. The last column is how this lab causes each failure without hurting anyone else on the machine.

| Inject | Expected | Usually observed | How this lab injects it (your containers only) |
|---|---|---|---|
| Pull the network | buffer, then drain | unbounded queue, OOM | `podman stop` your broker for 2 minutes at 40x the normal rate |
| Power cycle mid-write | clean restart | corrupt database file | `podman kill --signal KILL` both loggers, three times |
| Fill the storage | graceful degrade | silent data loss | fill each logger's 16 MB RAM disk to the last byte |
| Sensor returns NaN | flag and continue | crash loop | the sensor reads NaN for 12 seconds |
| Clock jumps backward | tolerate | duplicate or lost series | the logger's clock jumps back 2 minutes |
| Container OOM-killed | restart, resume | restarts, loses state | the logger eats memory until the 96 MB limit is hit |

### Why two loggers

Both loggers receive **exactly the same readings** from one sensor, so every drill is a fair comparison: same input, two designs. It also halves the time, because you see the naive and the hardened behavior in one run instead of two.

```text
                      udp/9000 (local link)
 sensor  ───────────────┬──────────────────────────┐
 5 readings/s           ▼                          ▼
                  logger-naive               logger-hardened
                  /data = 16 MB RAM disk     /data = 16 MB RAM disk
                  mem_limit 96 MB            mem_limit 96 MB
                        │ MQTT uplink              │ MQTT uplink
                        └──────────► broker ◄──────┘   (stands in for the cloud)
```

The sensor talks to the loggers over **UDP on the Compose network**, standing in for the I2C or serial bus a real sensor sits on, so stopping the broker cuts only the uplink, exactly like a field outage: the device keeps sensing while the cloud is unreachable.

<div class="uic-note">
<strong>New term · failure injection</strong>

Causing a failure on purpose, in a controlled way, to see whether the system handles it the way it was designed to. Large services do it in production (Netflix's Chaos Monkey kills servers at random during business hours); on an edge device you do it before deployment, because afterwards you cannot walk over and look.
</div>

---
## Part 3 · Build the Sensor

The sensor sends a reading 5 times a second to every logger on the local link: a sequence number, a temperature, and a 128-bin vibration spectrum (the kind of payload an accelerometer produces). Each logger says hello to the sensor twice a second, and the sensor sends to whoever said hello in the last 5 seconds; that way a logger that restarts on a new address is found again within half a second, and a dead one stops getting readings.

The drills switch two of the sensor's behaviors through `control/control.json`: a burst rate and a stuck sensor that reads NaN. Every switch has an end time, so a drill you interrupt switches itself off.

**[Notebook cell]** Create `app/sensor.py`:

In [ ]:
%%writefile app/sensor.py
import json
import math
import os
import random
import socket
import time

# The sensor talks to the loggers over the device's local link (UDP here,
# standing in for I2C or a serial bus). It never touches the broker.
loggerPort = int(os.environ.get("LOGGER_PORT", "9000"))
helloPort = int(os.environ.get("HELLO_PORT", "9001"))
controlPath = "/control/control.json"
statusPath = "/status/sensor.json"

controlCache = {"mtime": None, "data": {}}


def readControl():
    """Fault switches the notebook writes. Every fault has an end time, so a
    drill you interrupt switches itself off."""
    try:
        mtime = os.stat(controlPath).st_mtime_ns
        if mtime != controlCache["mtime"]:
            with open(controlPath) as controlFile:
                controlCache["data"] = json.load(controlFile)
            controlCache["mtime"] = mtime
    except (OSError, ValueError):
        pass
    return controlCache["data"]


link = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
link.bind(("0.0.0.0", helloPort))
link.setblocking(False)
listeners = {}            # logger address -> last time it said hello


def collectHellos():
    """Each logger says hello twice a second from wherever it is running now.
    A restarted logger comes back on a new address; its first hello tells us.
    (Looking the names up instead can hang for seconds on a stopped container.)"""
    while True:
        try:
            _, (address, _) = link.recvfrom(256)
        except (BlockingIOError, OSError):
            return
        if address not in listeners:
            print(f"logger at {address} said hello", flush=True)
        listeners[address] = time.monotonic()


seq = 0
nextSend = time.monotonic()
lastStatus = 0.0
print(f"sensor sending to every logger that says hello on udp/{helloPort}", flush=True)

while True:
    control = readControl()
    now = time.time()
    rate = control.get("burstRate", 5) if now < control.get("burstUntil", 0) else 5
    seq += 1
    temp = round(28 + 4 * math.sin(seq / 40) + random.uniform(-0.3, 0.3), 2)
    if now < control.get("nanUntil", 0):
        temp = float("nan")                      # a stuck sensor reads NaN
    reading = {"seq": seq, "temp": temp,
               "spectrum": [round(random.random(), 3) for _ in range(128)]}
    payload = json.dumps(reading).encode()
    collectHellos()
    for address, lastHello in list(listeners.items()):
        if time.monotonic() - lastHello > 5.0:
            del listeners[address]               # silent for 5 s: that logger is down
            print(f"logger at {address} went silent", flush=True)
            continue
        try:
            link.sendto(payload, (address, loggerPort))
        except OSError:
            pass                                 # the reading is gone for this logger
    if time.monotonic() - lastStatus >= 0.2:
        with open(statusPath + ".tmp", "w") as statusFile:
            json.dump({"seq": seq, "rate": rate, "time": now}, statusFile)
        os.replace(statusPath + ".tmp", statusPath)
        lastStatus = time.monotonic()
    nextSend += 1.0 / rate
    delay = nextSend - time.monotonic()
    if delay > 0:
        time.sleep(delay)
    elif delay < -1.0:
        nextSend = time.monotonic()              # fell behind: do not burst to catch up

<div class="uic-checkpoint">
<strong>Checkpoint · Part 3</strong>
</div>

In [ ]:
checkpoint(
    "Part 3 - sensor written",
    [
        check("sensor can be made to read NaN",
              fileContains("~/faultLab/app/sensor.py", 'temp = float("nan")'),
              hint="Re-run the %%writefile app/sensor.py cell in Part 3.",
              link="https://docs.python.org/3/library/math.html#math.nan",
              linkText="NaN in Python"),
        check("sensor sends to whichever loggers said hello",
              fileContains("~/faultLab/app/sensor.py", "def collectHellos():"),
              hint="Re-run the %%writefile app/sensor.py cell in Part 3.",
              link="https://docs.python.org/3/library/socket.html#socket.socket.recvfrom",
              linkText="socket.recvfrom"),
        check("sensor numbers every reading",
              fileContains("~/faultLab/app/sensor.py", '"seq": seq'),
              hint="Re-run the %%writefile app/sensor.py cell in Part 3 - the sequence number "
                   "is how the drills count lost readings.",
              link="https://docs.python.org/3/library/json.html", linkText="json module"),
    ],
    successNote="One sensor, numbered readings, and a switch for a stuck sensor.",
    docLink="https://docs.python.org/3/library/socket.html",
    docLinkText="Python socket module",
)

---
## Part 4 · Build the Logger: One Job, Two Builds

The logger receives readings, stores them in SQLite on its RAM disk, and uploads them over MQTT every 2 seconds when the uplink is up. `MODE=naive` and `MODE=hardened` differ only in the choices that matter when something fails:

| Concern | Naive (works on the bench) | Hardened (written for the field) |
|---|---|---|
| bad reading | stores whatever arrives | checks `math.isfinite`, flags NaN, keeps going |
| upload queue | a Python list in RAM, no limit | a table in SQLite, capped at 3000, drops the oldest and counts it |
| commits | every 100 readings, `synchronous=OFF` | every 0.5 s, `journal_mode=WAL` |
| write errors | `except sqlite3.OperationalError: pass` | logs it, frees space, never silent |
| full disk | does not notice | releases a 1 MB reserve, keeps the newest readings, drops the oldest, says so |
| timers | wall clock, `time.time()` | monotonic clock, `time.monotonic()` |
| after a restart | starts from zero | reloads its queue and counters from SQLite |

None of the naive choices is silly on its own. Each one is the obvious line to write, and each one passes every test on a bench where nothing fails.

**[Notebook cell]** Create `app/logger.py` (the bottom section, marked *lab instrumentation*, writes the status files and handles the OOM drill; it is not part of either design):

In [ ]:
%%writefile app/logger.py
import collections
import json
import math
import os
import socket
import sqlite3
import threading
import time

import paho.mqtt.client as mqtt

# One program, two builds. MODE=naive is code that works on the bench;
# MODE=hardened is the same job written for the field.
MODE = os.environ.get("MODE", "naive")
dbPath = f"/data/{MODE}.db"
reservePath = f"/data/{MODE}.reserve"
controlPath = "/control/control.json"
statusPath = f"/status/{MODE}.json"
uploadEvery = 2.0                                       # seconds between upload batches
maxPending = int(os.environ.get("MAX_PENDING", "3000"))  # hardened outbox limit
lowWater, highWater = 512 * 1024, 2 * 1024 * 1024       # bytes free on /data

stats = collections.Counter()
pending = []              # naive: the upload queue lives in RAM, no limit
control = {}


def log(message):
    print(f"[{MODE}] {message}", flush=True)


def wallNow():
    """This device's wall clock. The clock drill shifts it for a while."""
    if time.time() < control.get("clockUntil", 0):
        return time.time() + control.get("clockOffset", 0)
    return time.time()


# ---------------------------------------------------------------- storage
def openDatabase():
    db = sqlite3.connect(dbPath)
    if MODE == "naive":
        db.execute("PRAGMA synchronous=OFF")             # fast, and trusts the disk
        db.execute("CREATE TABLE IF NOT EXISTS readings "
                   "(ts REAL, seq INTEGER, temp REAL NOT NULL)")
    else:
        db.execute("PRAGMA journal_mode=WAL")            # crash-safe commits
        db.execute("PRAGMA synchronous=NORMAL")
        db.execute("PRAGMA wal_autocheckpoint=100")
        db.execute("CREATE TABLE IF NOT EXISTS readings "
                   "(ts REAL, seq INTEGER, temp REAL, flag TEXT)")
        db.execute("CREATE TABLE IF NOT EXISTS outbox (id INTEGER PRIMARY KEY, payload TEXT)")
        db.execute("CREATE TABLE IF NOT EXISTS counters (name TEXT PRIMARY KEY, value INTEGER)")
        stats.update(dict(db.execute("SELECT name, value FROM counters")))
        stats["boots"] += 1
    db.commit()
    return db


def storeNaive(db, reading):
    try:
        db.execute("INSERT INTO readings VALUES (?, ?, ?)",
                   (wallNow(), reading["seq"], reading["temp"]))
        stats["accepted"] += 1
        if stats["accepted"] % 100 == 0:
            db.commit()                                  # batch commits for speed
    except sqlite3.OperationalError:
        pass                                             # "the disk hiccuped", carry on
    pending.append(reading)                              # queue for upload, no limit


def storeHardened(db, reading, raw):
    temp = reading.get("temp")
    flag = None
    if not isinstance(temp, (int, float)) or not math.isfinite(temp):
        flag, temp = "nan", None                         # flag it and keep going
        stats["flagged"] += 1
    db.execute("INSERT INTO readings VALUES (?, ?, ?, ?)",
               (stampHardened(), reading["seq"], temp, flag))
    if stats["pendingNow"] >= maxPending:                # bounded queue: drop the oldest
        db.execute("DELETE FROM outbox WHERE id = (SELECT MIN(id) FROM outbox)")
        stats["dropped"] += 1
        stats["pendingNow"] -= 1
    db.execute("INSERT INTO outbox (payload) VALUES (?)", (raw,))
    stats["pendingNow"] += 1
    stats["accepted"] += 1


def commitHardened(db):
    try:
        db.executemany("INSERT OR REPLACE INTO counters VALUES (?, ?)", stats.items())
        db.commit()
    except sqlite3.OperationalError as error:            # never swallow it silently
        stats["writeErrors"] += 1
        log(f"WRITE FAILED ({error}): this batch is lost, freeing space")
        db.rollback()
        guardStorage(db, force=True)


def freeBytes():
    info = os.statvfs("/data")
    return info.f_bavail * info.f_frsize


def makeReserve():
    """Keep 1 MB of the disk as a spare tire, to be released when it fills."""
    with open(reservePath, "wb") as reserve:
        reserve.write(b"\0" * (1024 * 1024))


def guardStorage(db, force=False):
    free = freeBytes()
    if not stats["degraded"] and (free < lowWater or force):
        if os.path.exists(reservePath):
            os.remove(reservePath)                       # headroom for the cleanup below
        stats["degraded"] = 1
        stats["storageFull"] += 1
        stats["keepRows"] = db.execute("SELECT COUNT(*) FROM readings").fetchone()[0]
        log(f"STORAGE FULL ({free // 1024} KB free): keeping the newest "
            f"{stats['keepRows']} readings, dropping the oldest")
    if stats["degraded"]:
        excess = db.execute("SELECT COUNT(*) FROM readings").fetchone()[0] - stats["keepRows"]
        if excess > 0:                                   # stop growing: drop the oldest
            db.execute("DELETE FROM readings WHERE rowid IN "
                       "(SELECT rowid FROM readings ORDER BY rowid LIMIT ?)", (excess,))
            stats["trimmed"] += excess
        if free > highWater:                             # someone freed space
            makeReserve()
            stats["degraded"] = 0
            log("STORAGE OK again: reserve restored, growing normally")


# ---------------------------------------------------------------- clock
lastStamp = {"ts": None, "mono": None, "jumping": False}


def stampHardened():
    """Wall time for labels, but never earlier than the monotonic clock allows."""
    wall, mono = wallNow(), time.monotonic()
    ts = wall
    if lastStamp["ts"] is not None:
        floor = lastStamp["ts"] + (mono - lastStamp["mono"])
        if wall < floor - 1.0:                           # the wall clock stepped back
            ts = floor
            if not lastStamp["jumping"]:
                stats["clockJumps"] += 1
                log(f"CLOCK JUMP of {wall - floor:+.0f} s: stamping from the monotonic clock")
        lastStamp["jumping"] = ts != wall
    lastStamp["ts"], lastStamp["mono"] = ts, mono
    return ts


# ---------------------------------------------------------------- uplink
uplink = mqtt.Client(client_id=f"{socket.gethostname()}-{MODE}")
uplink.reconnect_delay_set(1, 2)
uplink.connect_async(os.environ.get("BROKER", "broker"), 1883, keepalive=5)  # notice a dead link fast
uplink.loop_start()


def uploadNaive():
    if uplink.is_connected():
        for reading in pending:
            uplink.publish(f"uplink/{MODE}", json.dumps(reading), qos=1)
        stats["uploaded"] += len(pending)
        pending.clear()


def uploadHardened(db):
    if not uplink.is_connected():
        return
    lastSent = None
    for rowId, payload in db.execute("SELECT id, payload FROM outbox ORDER BY id LIMIT 2000"):
        if uplink.publish(f"uplink/{MODE}", payload, qos=1).rc != mqtt.MQTT_ERR_SUCCESS:
            break
        lastSent = rowId
        stats["uploaded"] += 1
    if lastSent is not None:
        stats["pendingNow"] -= db.execute("DELETE FROM outbox WHERE id <= ?", (lastSent,)).rowcount


# ---------------------------------------------------------------- lab instrumentation
controlStamp = {"mtime": None}


def readControl():
    global control
    try:
        mtime = os.stat(controlPath).st_mtime_ns
        if mtime != controlStamp["mtime"]:
            with open(controlPath) as controlFile:
                control = json.load(controlFile)
            controlStamp["mtime"] = mtime
    except (OSError, ValueError):
        pass
    leakFlag = f"/control/leak-{MODE}"
    if os.path.exists(leakFlag):                         # the OOM drill
        os.remove(leakFlag)
        log("LEAK: allocating memory until the kernel steps in")
        hog = []
        while True:
            hog.append(b"x" * (8 * 1024 * 1024))


def memoryMB():
    try:
        with open("/sys/fs/cgroup/memory.current") as current:
            return round(int(current.read()) / 2**20, 1)
    except OSError:
        return None


sensorAddress = None


def keepFindingSensor():
    """Look the sensor's address up every 10 s (lab plumbing for the local
    link). On this network a lookup can stall for seconds while any container
    is stopped, so it has a thread of its own and the hellos never wait on it."""
    global sensorAddress
    while True:
        try:
            sensorAddress = socket.gethostbyname("sensor")
        except OSError:
            pass
        time.sleep(10 if sensorAddress else 1)


def keepSayingHello():
    """Tell the sensor where this logger is, twice a second."""
    while True:
        if sensorAddress:
            try:
                link.sendto(MODE.encode(), (sensorAddress, int(os.environ.get("HELLO_PORT", "9001"))))
            except OSError as error:
                log(f"hello to the sensor failed: {error}")
        time.sleep(0.5)


def writeStatus():
    status = {"mode": MODE, "time": time.time(), "connected": uplink.is_connected(),
              "pending": len(pending) if MODE == "naive" else stats["pendingNow"],
              "memMB": memoryMB(), "freeKB": freeBytes() // 1024, "firstSeq": firstSeq, **stats}
    with open(statusPath + ".tmp", "w") as statusFile:
        json.dump(status, statusFile)
    os.replace(statusPath + ".tmp", statusPath)


# ---------------------------------------------------------------- main loop
db = openDatabase()
if MODE == "hardened":
    stats["pendingNow"] = db.execute("SELECT COUNT(*) FROM outbox").fetchone()[0]
    if not os.path.exists(reservePath) and freeBytes() > highWater:
        makeReserve()
log(f"started: boot {stats['boots'] or 1}, {stats['pendingNow']} readings waiting to upload")

link = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
link.bind(("0.0.0.0", int(os.environ.get("LOGGER_PORT", "9000"))))
link.settimeout(0.1)
threading.Thread(target=keepFindingSensor, daemon=True).start()
threading.Thread(target=keepSayingHello, daemon=True).start()
lastUpload = wallNow() if MODE == "naive" else time.monotonic()
nextTick = time.monotonic()
firstSeq = 0                                             # first reading of this boot

while True:
    link.settimeout(0.1)
    try:
        while True:                                      # take every reading waiting
            raw, _ = link.recvfrom(65535)
            link.settimeout(0)
            reading = json.loads(raw)
            firstSeq = firstSeq or reading["seq"]
            if stats["lastSeq"] and reading["seq"] > stats["lastSeq"] + 1:
                stats["missed"] += reading["seq"] - stats["lastSeq"] - 1
            stats["lastSeq"] = reading["seq"]
            if MODE == "naive":
                storeNaive(db, reading)
            else:
                storeHardened(db, reading, raw.decode())
    except (socket.timeout, BlockingIOError):
        pass
    if MODE == "naive" and wallNow() - lastUpload >= uploadEvery:      # wall clock
        uploadNaive()
        lastUpload = wallNow()
    if MODE == "hardened" and time.monotonic() - lastUpload >= uploadEvery:  # monotonic
        uploadHardened(db)
        lastUpload = time.monotonic()
    if time.monotonic() >= nextTick:                     # twice a second
        readControl()
        if MODE == "hardened":
            guardStorage(db)
            commitHardened(db)
        writeStatus()
        nextTick = time.monotonic() + 0.5

**[Notebook cell]** One image serves the sensor and both loggers:

In [ ]:
%%writefile app/Dockerfile
FROM python:3.11-slim

WORKDIR /app

RUN pip install "paho-mqtt<2.0"

COPY sensor.py logger.py ./

CMD ["python", "logger.py"]

<div class="uic-checkpoint">
<strong>Checkpoint · Part 4</strong>
</div>

In [ ]:
checkpoint(
    "Part 4 - logger written",
    [
        check("hardened logger flags NaN instead of storing it",
              fileContains("~/faultLab/app/logger.py", "math.isfinite(temp)"),
              hint="Re-run the %%writefile app/logger.py cell in Part 4.",
              link="https://docs.python.org/3/library/math.html#math.isfinite",
              linkText="math.isfinite"),
        check("hardened upload queue is bounded",
              fileContains("~/faultLab/app/logger.py", 'if stats["pendingNow"] >= maxPending:'),
              hint="Re-run the %%writefile app/logger.py cell in Part 4.",
              link="https://docs.python.org/3/library/collections.html#collections.deque",
              linkText="bounded queues (deque maxlen)"),
        check("hardened timers use the monotonic clock",
              fileContains("~/faultLab/app/logger.py", "time.monotonic() - lastUpload >= uploadEvery"),
              hint="Re-run the %%writefile app/logger.py cell in Part 4.",
              link="https://docs.python.org/3/library/time.html#time.monotonic",
              linkText="time.monotonic"),
        check("hardened database uses write-ahead logging",
              fileContains("~/faultLab/app/logger.py", "PRAGMA journal_mode=WAL"),
              hint="Re-run the %%writefile app/logger.py cell in Part 4.",
              link="https://www.sqlite.org/wal.html", linkText="SQLite write-ahead logging"),
        check("Dockerfile pins paho-mqtt below 2.0",
              fileContains("~/faultLab/app/Dockerfile", "paho-mqtt<2.0"),
              hint="Re-run the %%writefile app/Dockerfile cell in Part 4.",
              link="https://docs.docker.com/reference/dockerfile/", linkText="Dockerfile reference"),
    ],
    successNote="Two builds of one logger: the drills will show where they part ways.",
    docLink="https://www.sqlite.org/atomiccommit.html",
    docLinkText="Atomic commit in SQLite",
)

---
## Part 5 · Wire It Up and Launch

**[Notebook cell]** Create the broker config and `compose.yaml`:

In [ ]:
%%writefile mosquitto/mosquitto.conf
listener 1883
allow_anonymous true

In [ ]:
%%writefile compose.yaml
services:
  broker:
    image: eclipse-mosquitto:2
    volumes:
      - ./mosquitto/mosquitto.conf:/mosquitto/config/mosquitto.conf

  sensor:
    build: ./app
    image: ${USER}-faults-app
    command: ["python", "sensor.py"]
    init: true
    restart: unless-stopped
    volumes:
      - ./control:/control
      - ./status:/status

  logger-naive:
    build: ./app
    image: ${USER}-faults-app
    command: ["python", "logger.py"]
    depends_on: [broker, disk]
    environment:
      MODE: naive
    dns_opt: ["timeout:1", "attempts:1"]
    init: true
    restart: unless-stopped
    mem_limit: 96m
    memswap_limit: 96m
    volumes:
      - naive-data:/data
      - ./control:/control
      - ./status:/status

  logger-hardened:
    build: ./app
    image: ${USER}-faults-app
    command: ["python", "logger.py"]
    depends_on: [broker, disk]
    environment:
      MODE: hardened
    dns_opt: ["timeout:1", "attempts:1"]
    init: true
    restart: unless-stopped
    mem_limit: 96m
    memswap_limit: 96m
    volumes:
      - hardened-data:/data
      - ./control:/control
      - ./status:/status

  disk:
    image: python:3.11-slim
    command: ["sleep", "infinity"]
    init: true
    volumes:
      - naive-data:/naive
      - hardened-data:/hardened

volumes:
  naive-data:
    driver_opts: {type: tmpfs, device: tmpfs, o: "size=16m"}
  hardened-data:
    driver_opts: {type: tmpfs, device: tmpfs, o: "size=16m"}

<div class="uic-note">
<strong>Note · what each setting is for</strong>

**`mem_limit: 96m` and `memswap_limit: 96m`:** each logger may use 96 MB of RAM and no swap. Go over and the kernel's OOM killer ends the process; the container exits with code **137** (128 + signal 9).

**`driver_opts: {type: tmpfs, ... size=16m}`:** each logger's `/data` is a 16 MB RAM disk of its own. Filling it fills nothing anyone else uses. It is RAM, so its contents vanish when nothing has it mounted.

**The `disk` service:** a container that only sleeps, with both RAM disks mounted. While it runs, the RAM disks stay mounted even when a logger dies, the way flash keeps its contents through a crash. It is also where you inspect the databases after a crash, like booting a rescue shell.

**`dns_opt: ["timeout:1", "attempts:1"]`:** on this Podman, looking up the name of a *stopped* container (the broker, during the outage drill) hangs for 20 seconds before it fails, and the MQTT client's reconnect waits on that lookup. One short attempt makes it fail fast, so the loggers reconnect within a couple of seconds of the broker coming back. A slow failure like this is a failure mode of its own: worth knowing about before the field finds it.

**`restart: unless-stopped` and `init: true`:** the same two settings as the *Fleet* lab. A logger that crashes is restarted at once; one you stop on purpose (`podman stop`, `podman kill`) stays down.
</div>

**[Notebook cell]** Write the lab instrumentation (`drills.py`). It reads the status files, asks the `disk` container what is really stored, and records each drill's result. You do not need to read it to do the lab:

In [ ]:
%%writefile drills.py
"""Lab 10 drill helpers: watch the two loggers, measure what happened, record it.

Every fault in this lab touches only YOUR containers (the ${USER}-faults
Compose project). Use it from a notebook cell (import drills) or from a
Jupyter terminal:  python3 ~/faultLab/drills.py watch
"""
import json
import os
import pathlib
import re
import subprocess
import sys
import time

labDir = pathlib.Path.home() / "faultLab"
project = os.environ.get("FAULT_PROJECT") or f"{os.environ.get('USER', 'student')}-faults"
loggers = ("naive", "hardened")
resultsDir = labDir / "results"
drillOrder = ["nan", "outage", "oom", "powercut", "diskfull", "clock"]
maxPending = 3000            # must match MAX_PENDING in logger.py


def container(service):
    return f"{project}-{service}-1"


def run(args, timeout=60, stdin=None):
    try:
        done = subprocess.run(args, capture_output=True, text=True, timeout=timeout, input=stdin)
        return done.returncode, done.stdout + done.stderr
    except subprocess.TimeoutExpired:
        return 124, f"timed out after {timeout}s"


# ------------------------------------------------------------------ fault switches
def setControl(**changes):
    """Write a fault switch the sensor and loggers read twice a second.
    Every switch carries an end time, so an interrupted drill turns itself off."""
    path = labDir / "control" / "control.json"
    try:
        data = json.loads(path.read_text())
    except (OSError, ValueError):
        data = {}
    data.update(changes)
    temp = path.with_name("control.json.tmp")
    temp.write_text(json.dumps(data))
    os.replace(temp, path)


def leak(name):
    """Ask one logger to allocate memory until the kernel OOM-kills it."""
    (labDir / "control" / f"leak-{name}").write_text("now\n")


def fillDisk():
    """Fill both 16 MB RAM disks to the last byte with a ballast file."""
    run(["podman", "exec", container("disk"), "sh", "-c",
         "dd if=/dev/zero of=/naive/ballast bs=1M 2>/dev/null;"
         "dd if=/dev/zero of=/hardened/ballast bs=1M 2>/dev/null; true"])


def freeDisk():
    run(["podman", "exec", container("disk"), "rm", "-f", "/naive/ballast", "/hardened/ballast"])


def resetFaults():
    """Turn every fault off and start every service. Safe to run any time."""
    setControl(burstUntil=0, nanUntil=0, clockUntil=0)
    for name in loggers:
        (labDir / "control" / f"leak-{name}").unlink(missing_ok=True)
    for service in ("disk", "broker", "sensor", "logger-naive", "logger-hardened"):
        run(["podman", "start", container(service)])
    freeDisk()
    print("all faults off; disk, broker, sensor and both loggers started")


# ------------------------------------------------------------------ measuring
def state(service):
    code, out = run(["podman", "inspect", container(service), "--format",
                     "{{.State.Status}}|{{.RestartCount}}"])
    if code != 0:
        return {"status": "missing", "restarts": 0}
    status, restarts = out.strip().split("|")
    return {"status": status, "restarts": int(restarts)}


def exitCodes(service, since):
    """Exit codes of every time this container died since a moment (podman events)."""
    code, out = run(["podman", "events", "--stream=false", "--since", str(int(since)),
                     "--filter", f"container={container(service)}", "--filter", "event=died",
                     "--format", "{{.ContainerExitCode}}"])
    return [int(word) for word in out.split() if word.lstrip("-").isdigit()]


def loggerStatus(name):
    try:
        status = json.loads((labDir / "status" / f"{name}.json").read_text())
    except (OSError, ValueError):
        return {}
    status["age"] = round(time.time() - status.get("time", 0), 1)
    return status


def sensorSeq():
    try:
        return json.loads((labDir / "status" / "sensor.json").read_text())["seq"]
    except (OSError, ValueError, KeyError):
        return 0


databaseProbe = r"""
import json, os, sqlite3, sys
since = int(sys.argv[1])
facts = {}
for name in ("naive", "hardened"):
    try:
        db = sqlite3.connect(f"/{name}/{name}.db", timeout=10)
        integrity = db.execute("PRAGMA integrity_check").fetchone()[0]
        rows, maxSeq = db.execute("SELECT COUNT(*), MAX(seq) FROM readings").fetchone()
        newer = db.execute("SELECT ts FROM readings WHERE seq > ? ORDER BY rowid", (since,)).fetchall()
        high = db.execute("SELECT MAX(ts) FROM readings WHERE seq <= ?", (since,)).fetchone()[0]
        overlap = 0
        for (ts,) in newer:
            if high is not None and ts is not None and ts <= high:
                overlap += 1
            if ts is not None and (high is None or ts > high):
                high = ts
        facts[name] = {"integrity": integrity, "rows": rows, "maxSeq": maxSeq or 0,
                       "newRows": len(newer), "overlap": overlap}
        db.close()
    except Exception as error:
        facts[name] = {"integrity": f"cannot open: {error}", "rows": 0, "maxSeq": 0,
                       "newRows": 0, "overlap": 0}
    disk = os.statvfs(f"/{name}")
    facts[name]["freeKB"] = disk.f_bavail * disk.f_frsize // 1024
print(json.dumps(facts))
"""


def databases(since=0):
    """Ask the disk container what is really stored: integrity, rows, newest seq."""
    code, out = run(["podman", "exec", "-i", container("disk"), "python3", "-", str(since)],
                    stdin=databaseProbe, timeout=60)
    try:
        return json.loads(out.strip().splitlines()[-1])
    except (ValueError, IndexError):
        return {name: {"integrity": out.strip()[:120], "rows": 0, "maxSeq": 0,
                       "newRows": 0, "overlap": 0, "freeKB": 0} for name in loggers}


def snapshot(withDatabases=False, since=0):
    snap = {"time": time.time(), "sensorSeq": sensorSeq()}
    for name in loggers:
        snap[name] = {**loggerStatus(name), **state(f"logger-{name}")}
    if withDatabases:
        snap["db"] = databases(since)
    return snap


def showRow(elapsed, name, entry, db=None):
    status = entry.get("status", "?")
    if status == "running" and entry.get("age", 99) > 3:
        status = "starting"
    line = (f"t+{elapsed:4.0f}s {name:<8} {status:<8} restarts {entry.get('restarts', 0):<3}"
            f" pending {entry.get('pending', 0):>5} uploaded {entry.get('uploaded', 0):>5}"
            f" dropped {entry.get('dropped', 0):>5} flagged {entry.get('flagged', 0):>3}"
            f" mem {entry.get('memMB') or 0:4.0f} MB")
    if db is not None:
        line += f" stored {db.get('rows', 0):>5} free {db.get('freeKB', 0):>5} KB"
    print(line, flush=True)


def observe(seconds, label, every=5, withDatabases=False, since=0):
    """Print both loggers every few seconds for a while; return the snapshots."""
    print(f"--- {label}: watching for {seconds} s", flush=True)
    start = time.time()
    snaps = []
    while True:
        snap = snapshot(withDatabases, since)
        snaps.append(snap)
        for name in loggers:
            showRow(time.time() - start, name, snap[name], snap.get("db", {}).get(name))
        if time.time() - start >= seconds - 0.5:
            return snaps
        time.sleep(max(0.0, min(every, seconds - (time.time() - start))))


def logsSince(service, since):
    return run(["podman", "logs", "--since", str(int(since)), container(service)])[1]


def delta(first, last, name, key):
    return (last[name].get(key) or 0) - (first[name].get(key) or 0)


# ------------------------------------------------------------------ results
def record(drill, inject, expected, naive, hardened, gap, metrics):
    resultsDir.mkdir(parents=True, exist_ok=True)
    entry = {"drill": drill, "inject": inject, "expected": expected, "naive": naive,
             "hardened": hardened, "gap": gap, "metrics": metrics, "time": time.time()}
    (resultsDir / f"{drill}.json").write_text(json.dumps(entry, indent=2))
    print(f"\nRECORDED {drill}\n  expected : {expected}\n  naive    : {naive}"
          f"\n  hardened : {hardened}\n  gap      : {gap}")


def result(drill):
    try:
        return json.loads((resultsDir / f"{drill}.json").read_text())
    except (OSError, ValueError):
        return None


def metric(drill, key, default=None):
    entry = result(drill)
    return entry["metrics"].get(key, default) if entry else default


def recordNan(before, snaps, since):
    after = snaps[-1]
    naiveRestarts = after["naive"]["restarts"] - before["naive"]["restarts"]
    hardenedRestarts = after["hardened"]["restarts"] - before["hardened"]["restarts"]
    flagged = delta(before, after, "hardened", "flagged")
    crashes = logsSince("logger-naive", since).count("IntegrityError")
    return record(
        "nan", "Sensor returns NaN", "flag and continue",
        f"crash loop: {naiveRestarts} restarts in {after['time'] - before['time']:.0f} s "
        f"(IntegrityError: SQLite stored NaN as NULL in a NOT NULL column)",
        f"flagged {flagged} NaN readings and kept running ({hardenedRestarts} restarts)",
        "naive: down for the whole fault and every reading in that window is gone; "
        "a sensor stuck for good means a crash loop for good",
        {"naiveRestarts": naiveRestarts, "hardenedRestarts": hardenedRestarts,
         "flagged": flagged, "integrityErrors": crashes})


def recordOutage(before, down, back, since):
    naiveExits = exitCodes("logger-naive", since)
    hardenedExits = exitCodes("logger-hardened", since)
    lostInRam, previous = 0, before
    for snap in down + back:                    # a restart wipes the queue it had
        if snap["naive"]["restarts"] > previous["naive"]["restarts"]:
            lostInRam += previous["naive"].get("pending", 0)
        previous = snap
    naivePeak = max(snap["naive"].get("pending", 0) for snap in down)
    naiveMem = max(snap["naive"].get("memMB") or 0 for snap in down)
    hardenedPeak = max(snap["hardened"].get("pending", 0) for snap in down)
    hardenedMem = max(snap["hardened"].get("memMB") or 0 for snap in down + back)
    dropped = delta(before, back[-1], "hardened", "dropped")
    drained = next((round(snap["time"] - back[0]["time"]) for snap in back
                    if snap["hardened"].get("pending", 0) < 300), None)
    seconds = round(down[-1]["time"] - down[0]["time"])
    oomKills = naiveExits.count(137)
    naiveText = (f"queue grew to {naivePeak} readings, {naiveMem:.0f} of 96 MB; "
                 + (f"OOM-killed {oomKills}x (exit 137), {lostInRam} queued readings lost"
                    if oomKills else "still growing when the uplink came back"))
    return record(
        "outage", f"Pull the network ({seconds} s at 200 readings/s)", "buffer, then drain",
        naiveText,
        f"queue held at {hardenedPeak} (dropped the {dropped} oldest, counted), "
        f"{hardenedMem:.0f} MB; drained "
        + (f"in {drained} s" if drained is not None else "not yet") + " after the uplink returned",
        "naive: memory grows with the outage until the kernel kills it; hardened keeps only "
        f"the newest {maxPending}, so a longer outage loses the oldest readings by design",
        {"naivePeakPending": naivePeak, "naivePeakMemMB": naiveMem, "naiveExitCodes": naiveExits,
         "naiveLostInRam": lostInRam, "hardenedPeakPending": hardenedPeak,
         "hardenedPeakMemMB": hardenedMem, "hardenedDropped": dropped,
         "hardenedExitCodes": hardenedExits, "drainSeconds": drained,
         "hardenedPendingAfter": back[-1]["hardened"].get("pending", 0)})


def bootLine(service, since):
    """The last 'started: boot N, Q readings waiting' line a logger printed."""
    found = re.findall(r"started: boot (\d+), (\d+) readings waiting", logsSince(service, since))
    return (int(found[-1][0]), int(found[-1][1])) if found else (0, None)


def recordOom(queued, since):
    exits = {name: exitCodes(f"logger-{name}", since) for name in loggers}
    queuedNaive = queued["naive"].get("pending", 0)
    queuedHardened = queued["hardened"].get("pending", 0)
    _, backNaive = bootLine("logger-naive", since)
    boots, backHardened = bootLine("logger-hardened", since)
    return record(
        "oom", "Container OOM-killed", "restart, resume",
        f"exit {exits['naive'][-1] if exits['naive'] else '?'}, restarted, came back with "
        f"{backNaive} of its {queuedNaive} queued readings and its counters at zero",
        f"exit {exits['hardened'][-1] if exits['hardened'] else '?'}, restarted as boot {boots} "
        f"with {backHardened} of its {queuedHardened} queued readings and resumed uploading",
        "naive: everything it held in RAM is gone after the restart; "
        "hardened: state lives in SQLite, so only work since the last commit (0.5 s) can be lost",
        {"naiveExitCodes": exits["naive"], "hardenedExitCodes": exits["hardened"],
         "naiveQueued": queuedNaive, "naiveAfter": backNaive if backNaive is not None else -1,
         "hardenedQueued": queuedHardened,
         "hardenedAfter": backHardened if backHardened is not None else -1, "hardenedBoots": boots})


def recordPowerCut():
    path = resultsDir / "powercut.jsonl"
    entries = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    lastRun = max(entry["run"] for entry in entries)
    entries = [entry for entry in entries if entry["run"] == lastRun]
    summary = {}
    for name in loggers:
        mine = [entry for entry in entries if entry["logger"] == name]
        summary[name] = {"cuts": len(mine),
                         "intact": sum(entry["integrity"] == "ok" for entry in mine),
                         "lost": [entry["lost"] for entry in mine]}
    def text(name):
        s = summary[name]
        return (f"integrity_check ok after {s['intact']} of {s['cuts']} cuts; "
                f"readings lost per cut: {', '.join(map(str, s['lost']))}")
    return record(
        "powercut", "Power cycle mid-write (podman kill -s KILL)", "clean restart",
        text("naive") + " (uncommitted batch of up to 100)",
        text("hardened") + " (commits every 0.5 s, WAL)",
        "a killed process cannot lose the OS page cache, so this test cannot show the "
        "corruption a real power cut can cause with synchronous=OFF; only pulling power "
        "on a real device closes that gap",
        summary)


def recordDiskFull(before, full, since):
    last = full[-1]
    accepted = delta(before, last, "naive", "accepted")
    storedNaive = last["db"]["naive"]["rows"] - before["db"]["naive"]["rows"]
    naiveLog = logsSince("logger-naive", since)
    naiveComplaints = sum(word in naiveLog.lower() for word in ("full", "error", "traceback"))
    hardenedLog = logsSince("logger-hardened", since)
    trimmed = delta(before, last, "hardened", "trimmed")
    staleNaive = max(0, last["sensorSeq"] - last["db"]["naive"]["maxSeq"])
    staleHardened = max(0, last["sensorSeq"] - last["db"]["hardened"]["maxSeq"])
    seconds = round(last["time"] - full[0]["time"])
    return record(
        "diskfull", "Fill the storage (16 MB RAM disk)", "graceful degrade",
        f"took {accepted} readings in {seconds} s and stored {max(storedNaive, 0)}; "
        f"no error in its log; newest stored reading is {staleNaive} readings old",
        f"logged STORAGE FULL, dropped the {trimmed} oldest to keep the newest; "
        f"newest stored reading is {staleHardened} readings old",
        "naive: silent data loss, nothing in the log says so; "
        "hardened: loses history on purpose and says so",
        {"naiveAccepted": accepted, "naiveStored": storedNaive,
         "naiveLogComplaints": naiveComplaints, "naiveStale": staleNaive,
         "hardenedStorageFullLogged": "STORAGE FULL" in hardenedLog,
         "hardenedTrimmed": trimmed, "hardenedStale": staleHardened})


def recordClock(before, during, after, jump):
    naiveUploads = delta(before, during[-1], "naive", "uploaded")
    hardenedUploads = delta(before, during[-1], "hardened", "uploaded")
    seconds = round(during[-1]["time"] - before["time"])
    return record(
        "clock", f"Clock jumps backward ({jump} s)", "tolerate",
        f"uploads stalled: {naiveUploads} sent in {seconds} s (wall-clock timer); "
        f"{after['db']['naive']['overlap']} stored rows overlap time already in the series",
        f"kept uploading ({hardenedUploads} sent) on the monotonic clock, logged the jump, "
        f"{after['db']['hardened']['overlap']} overlapping rows",
        "naive: a stall as long as the jump (a board that boots at 1970 waits 56 years) "
        "and a time series that folds back on itself",
        {"naiveUploads": naiveUploads, "hardenedUploads": hardenedUploads,
         "naiveOverlap": after["db"]["naive"]["overlap"],
         "hardenedOverlap": after["db"]["hardened"]["overlap"],
         "clockJumps": delta(before, during[-1], "hardened", "clockJumps")})


def failureTable():
    """All recorded drills, in slide order; also saved as results/failureTable.md."""
    rows = [result(drill) for drill in drillOrder if result(drill)]
    lines = ["| Inject | Expected | Naive: observed | Hardened: observed | Gap |",
             "|---|---|---|---|---|"]
    for row in rows:
        cells = [row[key].replace("|", "/") for key in
                 ("inject", "expected", "naive", "hardened", "gap")]
        lines.append("| " + " | ".join(cells) + " |")
    resultsDir.mkdir(parents=True, exist_ok=True)
    (resultsDir / "failureTable.md").write_text("\n".join(lines) + "\n")
    return rows


# ------------------------------------------------------------------ terminal entry points
def watch():
    print(f"Watching {project} every 3 s. Ctrl-C stops watching; the stack keeps running.")
    start = time.time()
    try:
        while True:
            snap = snapshot(withDatabases=True)
            print(f"sensor seq {snap['sensorSeq']}", flush=True)
            for name in loggers:
                showRow(time.time() - start, name, snap[name], snap["db"].get(name))
            time.sleep(3)
    except KeyboardInterrupt:
        print("\nstopped watching")


def powerCutCheck(runId, cut, seqAtCut):
    """Right after the kill: what survived on the disk? Lost = readings this
    boot had received (the sensor's count at the cut) but not yet committed."""
    facts = databases()
    resultsDir.mkdir(parents=True, exist_ok=True)
    with (resultsDir / "powercut.jsonl").open("a") as out:
        for name in loggers:
            stored = facts[name]["maxSeq"]
            firstSeq = loggerStatus(name).get("firstSeq") or seqAtCut
            lost = max(0, seqAtCut - max(stored, firstSeq - 1))
            entry = {"run": runId, "cut": cut, "logger": name, "integrity": facts[name]["integrity"],
                     "rows": facts[name]["rows"], "seqAtCut": seqAtCut, "stored": stored,
                     "lost": lost, "time": time.time()}
            out.write(json.dumps(entry) + "\n")
            print(f"  cut {cut} {name:<8} integrity_check: {entry['integrity']:<4} rows {entry['rows']:<6}"
                  f" newest stored seq {stored:<6} sensor was at {seqAtCut:<6} lost {entry['lost']}",
                  flush=True)


if __name__ == "__main__":
    command = sys.argv[1] if len(sys.argv) > 1 else "watch"
    if command == "watch":
        watch()
    elif command == "seq":
        print(sensorSeq())
    elif command == "check":
        powerCutCheck(int(sys.argv[2]), int(sys.argv[3]), int(sys.argv[4]))
    elif command == "reset":
        resetFaults()
    else:
        print(f"unknown command {command}")

**[Notebook cell]** Write the two terminal scripts and the initial fault switches (all off):

In [ ]:
%%writefile watchLoggers.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/faultLab/labEnv.sh
cd ~/faultLab

# Both loggers every 3 seconds: container state, restarts, queue, memory, rows
# really stored on disk. Ctrl-C stops watching; the stack keeps running.
python3 drills.py watch

In [ ]:
%%writefile powerCut.sh
#!/usr/bin/env bash
# Pull the plug on both loggers three times, in the middle of their writes.
set -uo pipefail
source ~/faultLab/labEnv.sh
cd ~/faultLab

runId=$(date +%s)
for cut in 1 2 3; do
  sleep $(( RANDOM % 8 + 5 ))                 # somewhere in the middle of the writes
  seqAtCut=$(python3 drills.py seq)           # where the sensor was at the moment of the cut
  echo "cut ${cut}: power off"
  podman kill --signal KILL "${FAULT_PROJECT}-logger-naive-1" "${FAULT_PROJECT}-logger-hardened-1"
  # what survived? PRAGMA integrity_check and a row count, from the disk container
  python3 drills.py check "${runId}" "${cut}" "${seqAtCut}"
  echo "cut ${cut}: power back on"
  podman start "${FAULT_PROJECT}-logger-naive-1" "${FAULT_PROJECT}-logger-hardened-1"
done
echo "done: three power cuts. Now run the record cell in Part 9 of the notebook."

In [ ]:
!chmod +x watchLoggers.sh powerCut.sh
!echo '{}' > control/control.json

**[Notebook cell]** Build the image and start the stack. `-d` detaches, so the cell returns; the build usually takes under a minute (the `python:3.11-slim` base is already in your image store from earlier labs, so only the `pip install` step runs, and often not even that). `--progress plain` prints one line per step instead of an animated display, which a notebook cannot redraw. Build lines like `level=warning msg="can't raise ambient capability ..."` are harmless: rootless Podman cannot hand those privileges to build steps, and this build does not need them:

In [ ]:
!podman compose --progress plain -p ${USER}-faults up --build -d

**[Notebook cell]** Confirm five containers are up, then take a first look at both loggers through the instrumentation:

In [ ]:
!podman compose -p ${USER}-faults ps
import time
import drills
time.sleep(5)
drills.observe(6, "first look: both loggers healthy", every=3, withDatabases=True)

**You should see:** five containers `Up` (`broker`, `sensor`, `disk`, `logger-naive`, `logger-hardened`), and both loggers `running` with `restarts 0`, a small `pending` count, a climbing `uploaded` count, and about 15 MB of memory. `stored` is what is really committed on disk: the naive logger's count moves in steps of 100 (its batch size), the hardened one's every half second.

**[Jupyter terminal]** Keep a live view of both loggers open in a terminal for the rest of the lab, next to this notebook:

```bash
~/faultLab/watchLoggers.sh
```

It prints both loggers every 3 seconds and never returns on its own; stop it with **Ctrl-C** (the stack keeps running).

<div class="uic-tip">
<strong>If a drill is interrupted</strong>

Every fault switch turns itself off after its drill's length, but a stopped broker or a full disk stays that way. If you interrupt a drill cell (the stop button) or something looks stuck, run <code>drills.resetFaults()</code> in a cell: it turns every fault off and starts every service.
</div>

<div class="uic-checkpoint">
<strong>Checkpoint · Part 5</strong>
</div>

In [ ]:
import os
userName = os.environ.get("USER", "student01")
services = ["broker", "sensor", "disk", "logger-naive", "logger-hardened"]

def stackRunning():
    namesOut, _ = runShell(["podman", "ps", "--format", "{{.Names}}"])
    up = [service for service in services if f"{userName}-faults-{service}-1" in namesOut.splitlines()]
    return len(up) == len(services), f"{len(up)}/{len(services)} running: {', '.join(up)}"

def loggersReporting():
    statuses = {name: drills.loggerStatus(name) for name in drills.loggers}
    fresh = [name for name, status in statuses.items() if status.get("age", 99) < 5]
    return len(fresh) == 2, f"reporting: {', '.join(fresh) or 'none'}"

def uplinkUp():
    connected = [name for name in drills.loggers if drills.loggerStatus(name).get("connected")]
    return len(connected) == 2, f"connected to the broker: {', '.join(connected) or 'none'}"

def memoryCapped():
    out, _ = runShell(["podman", "inspect", f"{userName}-faults-logger-naive-1",
                       f"{userName}-faults-logger-hardened-1", "--format", "{{.HostConfig.Memory}}"])
    limits = out.split()
    ok = len(limits) == 2 and all(limit == str(96 * 1024 * 1024) for limit in limits)
    return ok, f"memory limits (bytes): {', '.join(limits)}"

checkpoint(
    "Part 5 - stack running",
    [
        check("compose file is valid",
              commandSucceeds(f"cd ~/faultLab && podman compose -p {userName}-faults config --quiet"),
              hint="Run <code>podman compose config</code> in a Terminal from ~/faultLab to see "
                   "the YAML error, fix the compose cell, and re-run it.",
              link="https://docs.docker.com/reference/compose-file/",
              linkText="Compose file reference"),
        check("all five containers are running", stackRunning,
              hint="Re-run the compose up cell in Part 5, then check "
                   "<code>podman compose -p &lt;you&gt;-faults logs</code> in a Terminal. "
                   "If a drill was interrupted, run <code>drills.resetFaults()</code>.",
              link="https://docs.docker.com/reference/cli/docker/compose/up/",
              linkText="podman compose up reference"),
        check("both loggers are reporting", loggersReporting,
              hint="Wait ~10 seconds after startup and re-run this checkpoint.",
              link="https://docs.docker.com/reference/cli/docker/compose/logs/",
              linkText="podman compose logs reference"),
        check("both loggers reach the uplink", uplinkUp,
              hint="The broker may have started a moment after the loggers; they reconnect "
                   "within 2 seconds. Re-run this checkpoint.",
              link="https://mosquitto.org/", linkText="Eclipse Mosquitto"),
        check("each logger is capped at 96 MB", memoryCapped,
              hint="Re-run the %%writefile compose.yaml cell and the compose up cell.",
              link="https://docs.docker.com/reference/compose-file/services/#mem_limit",
              linkText="mem_limit"),
    ],
    successNote="Sensor, two loggers, broker and disk are up. Time to break things.",
    docLink="https://docs.podman.io/en/latest/markdown/podman-kill.1.html",
    docLinkText="podman kill",
)

---
## Part 6 · Drill 1: The Sensor Returns NaN

**NaN**, "not a number," is the floating-point value for an undefined result: 0/0, or a sensor driver that failed a read. Any arithmetic with NaN gives NaN, so one bad value poisons an average. Sensors return it more often than you would think: a loose wire, an I2C timeout, a reading taken while the sensor is still warming up.

**Expected:** flag it and continue. **Usually observed:** a crash loop.

**[Notebook cell]** Make the sensor read NaN for 12 seconds and watch both loggers. Keep an eye on the terminal too:

In [ ]:
import time

nanSeconds = 12                                       # how long the sensor stays stuck
nanStart = time.time()
nanBefore = drills.snapshot()
drills.setControl(nanUntil=time.time() + nanSeconds)  # the sensor now reads NaN
nanRun = drills.observe(nanSeconds + 10, "sensor stuck at NaN, then healthy again", every=4)
drills.recordNan(nanBefore, nanRun, nanStart)

**What happened:** the naive logger's `restarts` climbed by a few each second while the sensor was stuck. Look at why (it is in its log, `podman logs --tail 6 ${USER}-faults-logger-naive-1`):

```text
sqlite3.IntegrityError: NOT NULL constraint failed: readings.temp
```

Python happily parses `NaN` from JSON, and SQLite stores a NaN as **NULL**, which the naive table's `NOT NULL` column refuses. Nothing in the naive code catches an `IntegrityError` (it only catches `OperationalError`), so the process dies, the restart policy brings it back, the next reading is NaN again, and it dies again: a **crash loop**. Each restart also wiped its in-memory queue and counters. If the sensor stays broken, the loop never ends.

The hardened logger checked `math.isfinite(temp)` first, stored the row with `flag = 'nan'` and no temperature, and kept going. A gap you can see is better than a gap you cannot.

<div class="uic-note">
<strong>Note · restart loops and backoff</strong>

Podman restarts a crashed container **immediately, every time**. Docker waits a little longer after each crash, systemd gives up after a few quick failures (`StartLimitBurst`), and Kubernetes calls it `CrashLoopBackOff`. A restart policy is not a fix: it only decides how fast you fail.
</div>

<div class="uic-checkpoint">
<strong>Checkpoint · Part 6</strong>
</div>

In [ ]:
def nanMetric(key, default=0):
    return drills.metric("nan", key, default)

checkpoint(
    "Part 6 - NaN drill",
    [
        check("NaN drill recorded", lambda: (drills.result("nan") is not None, "results/nan.json"),
              hint="Run the drill cell in Part 6.",
              link="https://www.sqlite.org/nulls.html", linkText="NULL handling in SQLite"),
        check("naive logger crash-looped",
              lambda: (nanMetric("naiveRestarts") >= 1, f"{nanMetric('naiveRestarts')} restarts"),
              hint="The naive logger should restart at least once while the sensor reads NaN. "
                   "Check that the stack is up (Part 5 checkpoint) and re-run the drill cell.",
              link="https://docs.docker.com/engine/containers/start-containers-automatically/",
              linkText="Restart policies"),
        check("hardened logger flagged NaN and stayed up",
              lambda: (nanMetric("flagged") >= 1 and nanMetric("hardenedRestarts") == 0,
                       f"{nanMetric('flagged')} flagged, {nanMetric('hardenedRestarts')} restarts"),
              hint="Re-run the %%writefile app/logger.py cell, then the compose up cell in Part 5 "
                   "(it rebuilds), then the drill cell.",
              link="https://docs.python.org/3/library/math.html#math.isfinite",
              linkText="math.isfinite"),
    ],
    successNote="Same NaN, two outcomes: a crash loop and a flagged row.",
    docLink="https://www.sqlite.org/datatype3.html",
    docLinkText="SQLite datatypes",
)

---
## Part 7 · Drill 2: Pull the Network

In the field the uplink goes away for minutes or hours: a cellular dead zone, a router reboot, a backhoe. The device keeps sensing the whole time, so readings pile up waiting to be sent.

**Expected:** buffer, then drain. **Usually observed:** an unbounded queue, then an out-of-memory kill.

A real outage lasts much longer than this lab has, so the drill compresses it: the sensor runs at **200 readings a second** (40x normal) while **your** broker is stopped for **2 minutes**. Same queue growth as an 80-minute outage at the normal rate.

**[Notebook cell]** Stop the uplink, watch for 2 minutes, bring it back, and watch the drain (about 2.5 minutes in total):

In [ ]:
outageSeconds = 120                                   # field outages run for hours; 2 minutes shows the trend
outageStart = time.time()
outageBefore = drills.snapshot()
drills.setControl(burstRate=200, burstUntil=time.time() + outageSeconds + 30)   # 40x the normal rate
!podman stop ${USER}-faults-broker-1
outageDown = drills.observe(outageSeconds, "uplink down", every=10)
!podman start ${USER}-faults-broker-1
outageBack = drills.observe(30, "uplink back", every=5)
drills.setControl(burstUntil=0)                       # back to 5 readings a second
drills.recordOutage(outageBefore, outageDown, outageBack, outageStart)

**What happened:** the naive logger's `pending` and `mem` grew together, about 10 MB every 10 seconds, until the kernel killed it at the 96 MB limit (the `restarts` count went up and `pending` dropped to zero). Every reading it was holding for upload died with it. Then it started filling up again. At the field's normal rate the same thing happens, just later: this queue has no limit, so any outage long enough kills it.

The hardened logger's `pending` stopped at **3000** and its `dropped` counter started climbing: a **bounded queue** with a **drop-oldest** policy. Its memory stayed flat because the queue lives in SQLite on its disk, not in RAM. When the broker came back it drained the backlog in a few seconds.

Dropping data is not a failure here, it is the design: you chose what to lose (the oldest readings), you count it, and the device survives. The alternatives are a bigger cap, spilling to a bigger disk, or downsampling while offline; which one is right depends on what the data is for.

<div class="uic-checkpoint">
<strong>Checkpoint · Part 7</strong>
</div>

In [ ]:
def outageMetric(key, default=0):
    return drills.metric("outage", key, default)

checkpoint(
    "Part 7 - network outage drill",
    [
        check("outage drill recorded", lambda: (drills.result("outage") is not None, "results/outage.json"),
              hint="Run the drill cell in Part 7.",
              link="https://docs.podman.io/en/latest/markdown/podman-stop.1.html",
              linkText="podman stop"),
        check("naive queue grew past any limit",
              lambda: (outageMetric("naivePeakPending") > drills.maxPending,
                       f"peak {outageMetric('naivePeakPending')} queued"),
              hint="The naive queue should pass 3000 readings during the outage. Check that "
                   "the broker really stopped (the cell prints its name) and re-run the drill.",
              link="https://docs.python.org/3/tutorial/datastructures.html",
              linkText="Python lists"),
        check("hardened queue stayed bounded and counted its drops",
              lambda: (outageMetric("hardenedPeakPending") <= drills.maxPending + 50
                       and outageMetric("hardenedDropped") > 0,
                       f"peak {outageMetric('hardenedPeakPending')}, dropped {outageMetric('hardenedDropped')}"),
              hint="Re-run the %%writefile app/logger.py cell and the compose up cell in Part 5, "
                   "then the drill.",
              link="https://docs.python.org/3/library/collections.html#collections.deque",
              linkText="bounded queues"),
        check("hardened logger drained after the outage, never killed",
              lambda: (min(outageMetric("hardenedPendingAfter", 99999),
                           drills.loggerStatus("hardened").get("pending", 99999)) < 500
                       and 137 not in outageMetric("hardenedExitCodes", []),
                       f"{drills.loggerStatus('hardened').get('pending')} queued now, "
                       f"exits {outageMetric('hardenedExitCodes', [])}"),
              hint="If it is still draining, wait 20 seconds and re-run this checkpoint. "
                   "Check that the broker is running again (<code>podman ps</code>); run "
                   "<code>drills.resetFaults()</code> and re-run the drill if not.",
              link="https://docs.podman.io/en/latest/markdown/podman-start.1.html",
              linkText="podman start"),
    ],
    successNote="Buffer, then drain - and you know exactly what was dropped.",
    docLink="https://www.eclipse.org/paho/index.php?page=clients/python/docs/index.php",
    docLinkText="Paho MQTT Python client",
)

---
## Part 8 · Drill 3: The Container Is OOM-Killed

You saw the naive logger die of its own queue. Memory runs out for other reasons too: a leak, a big image, a neighbor process. The question in this drill is not whether the logger restarts (the restart policy takes care of that) but **what it remembers when it comes back**.

**Expected:** restart, resume. **Usually observed:** restarts, loses state.

**[Notebook cell]** Stop the uplink for 30 seconds so both loggers hold a queue, then make each logger allocate memory until the kernel kills it, and watch them come back (about 1 minute):

In [ ]:
oomStart = time.time()
!podman stop ${USER}-faults-broker-1
drills.observe(30, "uplink down: both loggers queue readings", every=10)
oomQueued = drills.snapshot()
drills.leak("naive")                                  # each logger now eats memory
drills.leak("hardened")                               # until the kernel steps in
drills.observe(12, "OOM kill and automatic restart", every=3)
!podman start ${USER}-faults-broker-1
drills.observe(12, "uplink back", every=4)
drills.recordOom(oomQueued, oomStart)

**What happened:** both containers died with exit code **137** (the kernel's `SIGKILL`; `podman events` records it) and both were restarted at once. The naive logger came back with an empty queue and its counters at zero: the readings it was holding for upload are gone, and nothing anywhere records that they existed. The hardened logger came back as a new boot with its queue intact, reloaded from SQLite, and uploaded it when the broker returned.

The rule: anything you cannot afford to lose must be written somewhere that outlives the process, and committed often enough that the window you can lose is small (here, half a second).

<div class="uic-checkpoint">
<strong>Checkpoint · Part 8</strong>
</div>

In [ ]:
def oomMetric(key, default=0):
    return drills.metric("oom", key, default)

checkpoint(
    "Part 8 - OOM drill",
    [
        check("OOM drill recorded", lambda: (drills.result("oom") is not None, "results/oom.json"),
              hint="Run the drill cell in Part 8.",
              link="https://docs.podman.io/en/latest/markdown/podman-events.1.html",
              linkText="podman events"),
        check("both loggers were OOM-killed (exit 137)",
              lambda: (137 in oomMetric("naiveExitCodes", []) and 137 in oomMetric("hardenedExitCodes", []),
                       f"naive {oomMetric('naiveExitCodes', [])}, hardened {oomMetric('hardenedExitCodes', [])}"),
              hint="The leak needs the 96 MB limit from compose.yaml. Re-run the Part 5 "
                   "checkpoint, then this drill.",
              link="https://docs.docker.com/reference/compose-file/services/#mem_limit",
              linkText="mem_limit"),
        check("hardened logger resumed its queue; naive lost its own",
              lambda: (oomMetric("hardenedAfter", -1) > 0 and oomMetric("naiveAfter", -1) == 0,
                       f"hardened came back with {oomMetric('hardenedAfter')}, "
                       f"naive with {oomMetric('naiveAfter')}"),
              hint="Both loggers need a queue when they are killed: check that the broker was "
                   "stopped during the first 30 seconds, then re-run the drill.",
              link="https://www.sqlite.org/atomiccommit.html", linkText="Atomic commit in SQLite"),
    ],
    successNote="Restart is the easy half. Resuming is the part you design.",
    docLink="https://www.kernel.org/doc/html/latest/admin-guide/cgroup-v2.html#memory",
    docLinkText="cgroup v2 memory controller",
)

---
## Part 9 · Drill 4: Power Cut Mid-Write

A power cut stops everything between one instruction and the next. A database that is half way through writing a change has to recover from whatever made it to storage.

**Expected:** clean restart. **Usually observed:** a corrupt database file.

You cannot pull the power on the DGX Spark. The closest you can get is killing the process with `SIGKILL`, which gives it no chance to finish, flush, or clean up. `powerCut.sh` does that to both loggers three times at random moments, checks each database right after the cut with SQLite's `PRAGMA integrity_check` from the `disk` container, counts what was lost, and turns the power back on.

**[Jupyter terminal]** Run the power-cut drill (about 40 seconds) and watch the live view in your other terminal while it runs:

```bash
~/faultLab/powerCut.sh
```

**You should see:** for each cut, one line per logger: `integrity_check: ok`, the row count, and how many readings were **lost**: received but not yet committed when the power went. The naive logger loses up to 100 per cut (its batch); the hardened logger loses 0 to 3 (it commits every half second).

<div class="uic-note">
<strong>Note · what this drill cannot show</strong>

This is **weaker than a real power cut**. Killing a process does not kill the operating system: everything the process already handed to the kernel is still in the page cache and still reaches the disk. A real power cut loses the page cache too. That is where `synchronous=OFF` (the naive setting) and cheap SD cards that ignore `fsync` turn into corrupt database files, and no `podman kill` can reproduce it. The only honest test is pulling power on a real device, many times. Write that limit into your report: it is part of the gap.
</div>

**[Notebook cell]** After `powerCut.sh` prints `done`, record the result:

In [ ]:
drills.recordPowerCut()

<div class="uic-checkpoint">
<strong>Checkpoint · Part 9</strong>
</div>

In [ ]:
def powerCutSummary(name):
    summary = drills.result("powercut")
    return summary["metrics"].get(name, {}) if summary else {}

checkpoint(
    "Part 9 - power-cut drill",
    [
        check("three power cuts recorded",
              lambda: (powerCutSummary("hardened").get("cuts", 0) >= 3,
                       f"{powerCutSummary('hardened').get('cuts', 0)} cut(s)"),
              hint="Run <code>~/faultLab/powerCut.sh</code> in a Jupyter terminal, wait for "
                   "<code>done</code>, then run the record cell above.",
              link="https://docs.podman.io/en/latest/markdown/podman-kill.1.html",
              linkText="podman kill"),
        check("hardened database intact after every cut",
              lambda: (powerCutSummary("hardened").get("intact", 0) == powerCutSummary("hardened").get("cuts", -1),
                       f"{powerCutSummary('hardened').get('intact', 0)} of "
                       f"{powerCutSummary('hardened').get('cuts', 0)} ok"),
              hint="A failed integrity check on the hardened logger is a real finding; tell "
                   "your instructor. Re-run powerCut.sh to see whether it repeats.",
              link="https://www.sqlite.org/pragma.html#pragma_integrity_check",
              linkText="PRAGMA integrity_check"),
        check("both loggers running again",
              lambda: (all(drills.state(f"logger-{name}")["status"] == "running" for name in drills.loggers),
                       ", ".join(f"{name} {drills.state(f'logger-{name}')['status']}" for name in drills.loggers)),
              hint="powerCut.sh restarts both loggers at the end of each cut; if it was "
                   "interrupted, run <code>drills.resetFaults()</code>.",
              link="https://docs.podman.io/en/latest/markdown/podman-start.1.html",
              linkText="podman start"),
    ],
    successNote="Clean restarts after every cut - and you know what this test cannot prove.",
    docLink="https://www.sqlite.org/howtocorrupt.html",
    docLinkText="How to corrupt an SQLite database file",
)

---
## Part 10 · Drill 5: Fill the Storage

Edge storage fills up: logs nobody rotates, a database nobody trims, a camera that records more than planned. When it is full, every write fails.

**Expected:** graceful degrade. **Usually observed:** silent data loss.

**[Notebook cell]** Fill both 16 MB RAM disks to the last byte with a ballast file, watch for 45 seconds, then take the ballast back out (about 1 minute). The `No space left on device` lines from `dd` are the point:

In [ ]:
diskStart = time.time()
diskBefore = drills.snapshot(withDatabases=True)
!podman exec ${USER}-faults-disk-1 dd if=/dev/zero of=/naive/ballast bs=1M
!podman exec ${USER}-faults-disk-1 dd if=/dev/zero of=/hardened/ballast bs=1M
!podman exec ${USER}-faults-disk-1 df -h /naive /hardened
diskFull = drills.observe(45, "storage full", every=15, withDatabases=True)
drills.recordDiskFull(diskBefore, diskFull, diskStart)
!podman exec ${USER}-faults-disk-1 rm -f /naive/ballast /hardened/ballast
drills.observe(6, "ballast removed", every=6, withDatabases=True)

**What happened:** the naive logger kept running, kept counting readings as accepted, and printed nothing. But its `stored` count did not move: every batch commit failed with `database or disk is full`, and its `except sqlite3.OperationalError: pass` threw the error and the batch away. From the outside it looked healthy; its newest stored reading just kept getting older. That is **silent data loss**, and it is how a device can run for weeks storing nothing.

The hardened logger logged `STORAGE FULL`, released its 1 MB reserve file (a spare tire, set aside for exactly this), and switched to keeping the newest readings and dropping the oldest. Its newest stored reading stayed current. When the ballast came out it logged `STORAGE OK` and went back to growing.

The trick worth stealing: reserve a little space you can give back, so the code that cleans up after a full disk has room to run. Linux file systems do the same for root (the 5% `ext4` reserve).

<div class="uic-checkpoint">
<strong>Checkpoint · Part 10</strong>
</div>

In [ ]:
def diskMetric(key, default=0):
    return drills.metric("diskfull", key, default)

def diskSpaceBack():
    facts = drills.databases()
    free = {name: facts[name].get("freeKB", 0) for name in drills.loggers}
    return all(value > 8000 for value in free.values()), f"free KB: {free}"

checkpoint(
    "Part 10 - disk-full drill",
    [
        check("disk-full drill recorded", lambda: (drills.result("diskfull") is not None, "results/diskfull.json"),
              hint="Run the drill cell in Part 10.",
              link="https://www.sqlite.org/rescode.html#full", linkText="SQLITE_FULL"),
        check("naive logger lost readings without a word",
              lambda: (diskMetric("naiveAccepted") - diskMetric("naiveStored") >= 50
                       and diskMetric("naiveLogComplaints") == 0,
                       f"accepted {diskMetric('naiveAccepted')}, stored {diskMetric('naiveStored')}, "
                       f"log lines about it {diskMetric('naiveLogComplaints')}"),
              hint="Both RAM disks must be full for the whole 45 seconds; re-run the drill cell.",
              link="https://docs.python.org/3/library/sqlite3.html#sqlite3.OperationalError",
              linkText="sqlite3.OperationalError"),
        check("hardened logger said so and kept the newest readings",
              lambda: (diskMetric("hardenedStorageFullLogged", False) and diskMetric("hardenedStale", 999) < 30,
                       f"logged {diskMetric('hardenedStorageFullLogged', False)}, newest stored "
                       f"reading {diskMetric('hardenedStale', 999)} readings old"),
              hint="Re-run the %%writefile app/logger.py cell and the compose up cell in Part 5, "
                   "then the drill.",
              link="https://man7.org/linux/man-pages/man3/statvfs.3.html", linkText="statvfs"),
        check("ballast removed, space back", diskSpaceBack,
              hint="Run <code>drills.freeDisk()</code> in a cell.",
              link="https://man7.org/linux/man-pages/man1/df.1.html", linkText="df"),
    ],
    successNote="Same full disk: one logger lied by omission, the other degraded on purpose.",
    docLink="https://www.sqlite.org/rescode.html#full",
    docLinkText="SQLITE_FULL",
)

---
## Part 11 · Drill 6: The Clock Jumps Backward

Device clocks jump. NTP corrects a drifting clock in one step; a board without a battery-backed clock boots believing it is 1970 and then jumps forward 56 years when it reaches a time server; someone fixes the time zone by hand.

**Expected:** tolerate. **Usually observed:** duplicate or lost series.

You cannot change the DGX Spark's clock (and on a shared machine you would not want to), so the drill shifts the time **inside both loggers**: for 45 seconds their `wallNow()` runs 2 minutes behind.

**[Notebook cell]** Jump the loggers' clock back 2 minutes and watch for 40 seconds:

In [ ]:
clockJump = -120                                      # seconds; NTP steps and RTC-less boots do this
clockSince = drills.sensorSeq()
clockBefore = drills.snapshot()
drills.setControl(clockOffset=clockJump, clockUntil=time.time() + 45)
clockDuring = drills.observe(40, "clock jumped back 2 minutes", every=10)
drills.setControl(clockUntil=0)                       # the clock is right again
time.sleep(3)
clockAfter = drills.snapshot(withDatabases=True, since=clockSince)
drills.recordClock(clockBefore, clockDuring, clockAfter, clockJump)

**What happened:** the naive logger's `uploaded` count froze and its `pending` grew. Its upload timer is `wallNow() - lastUpload >= 2`: after the clock went back 2 minutes that difference is negative, and it stays below 2 seconds until the clock catches up with where it was, 2 minutes later. Meanwhile the timestamps it stored went back in time, overlapping readings it had already stored: a dashboard grouping by time would show duplicates, and anything keyed by timestamp could overwrite good data.

The hardened logger runs its timers on `time.monotonic()`, which never goes backward, so its uploads never paused. It noticed the wall clock stepping back (`CLOCK JUMP` in its log) and stamped readings from the last good time plus the monotonic time elapsed, so its series never folded back on itself.

The rule: **monotonic clock for intervals and timeouts, wall clock only for labels**, and never trust a label without checking it against the monotonic clock.

<div class="uic-checkpoint">
<strong>Checkpoint · Part 11</strong>
</div>

In [ ]:
def clockMetric(key, default=0):
    return drills.metric("clock", key, default)

checkpoint(
    "Part 11 - clock-jump drill",
    [
        check("clock drill recorded", lambda: (drills.result("clock") is not None, "results/clock.json"),
              hint="Run the drill cell in Part 11.",
              link="https://docs.python.org/3/library/time.html#time.monotonic",
              linkText="time.monotonic"),
        check("naive uploads stalled and its series overlapped",
              lambda: (clockMetric("naiveUploads", 1) == 0 and clockMetric("naiveOverlap") > 0,
                       f"{clockMetric('naiveUploads', 1)} uploads, {clockMetric('naiveOverlap')} overlapping rows"),
              hint="The jump must last the whole 40 seconds; re-run the drill cell.",
              link="https://docs.python.org/3/library/time.html#time.time",
              linkText="time.time"),
        check("hardened kept uploading with a clean series",
              lambda: (clockMetric("hardenedUploads") > 0 and clockMetric("hardenedOverlap", 1) == 0,
                       f"{clockMetric('hardenedUploads')} uploads, {clockMetric('hardenedOverlap', 1)} overlapping rows"),
              hint="Re-run the %%writefile app/logger.py cell and the compose up cell in Part 5, "
                   "then the drill.",
              link="https://www.python.org/dev/peps/pep-0418/", linkText="PEP 418: monotonic clocks"),
    ],
    successNote="Monotonic for intervals, wall clock for labels.",
    docLink="https://docs.python.org/3/library/time.html#time.monotonic",
    docLinkText="time.monotonic",
)

---
## Part 12 · The Expected / Observed / Gap Table

This is the deliverable. The slide said it: run each failure deliberately, record what happened, and put the table in your final report. Every row below comes from measurements your drills just made, not from the lecture.

**[Notebook cell]** Print the table and save it as Markdown:

In [ ]:
rows = drills.failureTable()
table = Table(title="Failure injection: expected vs observed", box=box.SIMPLE_HEAVY, show_lines=True)
for column in ("Inject", "Expected", "Naive: observed", "Hardened: observed", "Gap"):
    table.add_column(column, overflow="fold")
for row in rows:
    table.add_row(row["inject"], row["expected"], row["naive"], row["hardened"], row["gap"])
richConsole.print(table)
print(f"{len(rows)} drill(s) recorded; the table is saved as ~/faultLab/results/failureTable.md")

### Use it for Milestone 2 and your final report

Your own project is the system that matters. Take this table's shape to your node:

1. Pick at least two rows that can happen to **your** deployment.
2. Write down the expected behavior **before** you inject anything.
3. Inject it on purpose, on your own containers or your own device, and record what happened, with numbers: restarts, readings lost, seconds to recover.
4. The **gap** column is the honest part: where your system fell short, and what this kind of test cannot show (like the page cache in the power-cut drill).

Milestone 2 asks what broke. This table is how you answer with evidence instead of a story.

<div class="uic-checkpoint">
<strong>Checkpoint · Part 12</strong>
</div>

In [ ]:
def tableRows():
    rows = drills.failureTable()
    return len(rows) >= 4, f"{len(rows)} drill(s) in results/failureTable.md"

checkpoint(
    "Part 12 - failure table",
    [
        check("at least four drills in the table", tableRows,
              hint="Run the drill cells in Parts 6 to 11 (Part 9 needs powerCut.sh in a terminal), "
                   "then re-run the table cell.",
              link="https://www.sqlite.org/howtocorrupt.html",
              linkText="How to corrupt an SQLite database file"),
        check("table saved as Markdown", fileNonEmpty("~/faultLab/results/failureTable.md", minLines=6),
              hint="Re-run the table cell above.",
              link="https://www.markdownguide.org/extended-syntax/#tables",
              linkText="Markdown tables"),
    ],
    successNote="Expected, observed, gap: the evidence your final report needs.",
    docLink="https://www.markdownguide.org/extended-syntax/#tables",
    docLinkText="Markdown tables",
)

---
## Cleanup

**[Jupyter terminal]** Stop the live view with **Ctrl-C** if it is still running.

**[Notebook cell]** Stop and remove the whole stack, including its network and both RAM disks, remove the image Compose built, then confirm nothing of yours is left running (safe to run any number of times). Your results in `~/faultLab/results` stay:

In [ ]:
%cd ~/faultLab
!podman compose --progress plain -p ${USER}-faults down -v
!podman rmi ${USER}-faults-app 2>/dev/null || true
!podman compose -p ${USER}-faults ps

**[Notebook cell]** Optional · remove your lab folder (uncomment to run). Copy `results/failureTable.md` somewhere first if you want it for your report. Only remove your own files, containers, networks, and volumes.

In [ ]:
# !rm -rf ~/faultLab

### Lab scorecard

Run this any time for a live summary of every checkpoint in this kernel session. If a row is incomplete, scroll up, fix that part, re-run its checkpoint, and re-run this cell.

In [ ]:
labSummary("Failure Injection")

---
## Connect to the Rest of the Course

Every earlier lab built something that works. This one asks how it fails:

```text
Telemetry + SQLite      -> what a power cut and a full disk do to stored data
MQTT                    -> what an outage does to the upload path
Podman + Compose        -> memory limits, restart policies, exit code 137
Fleet monitoring        -> a heartbeat sees a crash loop; it cannot see silent data loss
Benchmarking            -> numbers with units: restarts, readings lost, seconds to drain
```

Real deployments hit every row of the table. Connected vehicles drive through dead zones, solar-powered sensors brown out mid-write, and cameras fill their cards. The difference between a demo and a product is whether someone broke it on purpose first.

---
## Key Terms

- **Failure injection:** causing a failure on purpose, in a controlled way, to check that the system handles it as designed.
- **Crash loop:** a process that crashes, is restarted, hits the same input, and crashes again, indefinitely.
- **Bounded queue:** a buffer with a maximum size and a policy for when it is full (drop oldest, drop newest, or spill to disk with a size cap).
- **OOM kill:** the kernel ending a process that went over its memory limit; a container reports exit code 137.
- **Write-ahead log (WAL):** SQLite's journaling mode that keeps the database consistent across crashes, provided the storage honors `fsync`.
- **`PRAGMA integrity_check`:** SQLite's full consistency check of a database file; `ok` means no corruption was found.
- **Graceful degradation:** losing capability on purpose and visibly (keep the newest data, drop the oldest, say so) instead of failing silently.
- **Monotonic clock:** a clock that never goes backward (`time.monotonic()`); use it for intervals, and wall-clock time only for labels.
- **NaN:** "not a number," the floating-point value for undefined results; SQLite stores it as NULL.

---
### One-minute feedback

Your feedback shapes the next version of this lab. Rate it, add anything that was confusing or broken, and click **Submit**. It takes about 30 seconds and goes straight to the instructor.

In [ ]:
feedback("Failure Injection")